# Feature engineering — Billings
This notebook creates simple, easy-to-follow features from the processed billings file. The code is split into small cells so a beginner can follow along.

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [2]:
df = pd.read_csv("../../data/02_processed/processed_billings.csv")
df.head()

C:\Users\NuluShreya\AppData\Local\Temp\ipykernel_13896\678099606.py:1: DtypeWarning: Columns (0: proforma_auto_renewal, 1: proforma_world_pay_token, 2: current_anchor_list, 3: payment_timeframe, 4: proforma_approved_lists, 5: last_band) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/02_processed/processed_billings.csv")


,co_ref,renewal_month,discount_amount,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,...,last_renewal,last_band,last_total_net_paid,last_connections,anchor_group,renewal_year,datetime_out,registration_date_missing_flag,prospect_renewal_date_missing_flag,proforma_date_missing_flag
0,VT6174,2024-01-11,NaN,8.0,42.5,799.0,9,9,7.5,9.0,...,2023-01-11 00:00:00,Band B,664.0,1.0,1,2024,2024-01-11,0,0,0
1,VD3828,2025-01-08,NaN,8.0,41.5,799.0,9,9,7.5,8.0,...,No_History,NaN,0.0,0.0,1,2025,2025-01-08,0,0,0
2,DV8120,2025-01-03,NaN,8.0,33.0,799.0,8,0,7.5,9.5,...,2024-01-03 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-03,0,0,0
3,EZ9894,2025-01-06,NaN,9.5,44.5,799.0,9,9,7.5,9.5,...,2024-01-06 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-06,1,1,1
4,FA8957,2025-01-03,NaN,9.5,42.5,799.0,9,8,7.5,8.5,...,2024-01-03 00:00:00,Band C1,749.0,1.0,1,2025,2025-01-03,1,1,0


In [3]:
df['prospect_renewal_date'] = pd.to_datetime(
    df['prospect_renewal_date'], errors='coerce'
)

df['datetime_out'] = pd.to_datetime(
    df['datetime_out'], errors='coerce'
)

In [4]:
df = df.dropna(subset=['prospect_renewal_date', 'datetime_out'])

# Keep only closed outcomes for supervised modeling target
df['prospect_outcome'] = df['prospect_outcome'].astype(str).str.strip().str.title()
df = df[df['prospect_outcome'].isin(['Won', 'Churned'])]

df['cutoff_date'] = df['prospect_renewal_date'] - timedelta(days=14)

In [5]:
df['prospect_renewal_date'].head()


0   2024-05-11
1   2025-09-08
2   2025-12-03
7   2024-08-11
9   2025-04-04
Name: prospect_renewal_date, dtype: datetime64[us]

In [6]:
df = df[df['datetime_out'] <= df['cutoff_date']]

In [7]:
agg_df = df.groupby('co_ref').agg({
    'amount': ['sum', 'mean', 'count'],
    'datetime_out': ['max']
}).reset_index()

agg_df.columns = [
    'co_ref',
    'total_spent',
    'avg_payment',
    'num_payments',
    'last_payment_date'
]

In [8]:
cutoff_map = df[['co_ref', 'cutoff_date']].drop_duplicates()

agg_df = agg_df.merge(cutoff_map, on='co_ref', how='left')

agg_df['days_since_last_payment'] = (
    agg_df['cutoff_date'] - agg_df['last_payment_date']
).dt.days

In [9]:
df['days_before_cutoff'] = (df['cutoff_date'] - df['datetime_out']).dt.days

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 30].groupby('co_ref')['amount'].count().rename('payments_last_30'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 90].groupby('co_ref')['amount'].count().rename('payments_last_90'),
    on='co_ref', how='left'
)

In [10]:
agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 30].groupby('co_ref')['amount'].sum().rename('spend_last_30'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df[df['days_before_cutoff'] <= 90].groupby('co_ref')['amount'].sum().rename('spend_last_90'),
    on='co_ref', how='left'
)

In [11]:
last_30 = df[df['days_before_cutoff'] <= 30].groupby('co_ref')['amount'].sum()

prev_30 = df[
    (df['days_before_cutoff'] > 30) & (df['days_before_cutoff'] <= 60)
].groupby('co_ref')['amount'].sum()

trend = (last_30 - prev_30).rename('payment_trend')

agg_df = agg_df.merge(trend, on='co_ref', how='left')

In [12]:
# Tenure
agg_df = agg_df.merge(
    df[['co_ref', 'tenure_years']].drop_duplicates(),
    on='co_ref', how='left'
)

# Payment method (mode)
payment_mode = df.groupby('co_ref')['payment_method'].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown'
)

agg_df = agg_df.merge(payment_mode.rename('payment_method_mode'), on='co_ref', how='left')

# Target label: take the latest known prospect_outcome per customer
label_df = (
    df.sort_values('prospect_renewal_date')
      .dropna(subset=['prospect_outcome'])
      .drop_duplicates(subset=['co_ref'], keep='last')[['co_ref', 'prospect_outcome']]
      .copy()
)

agg_df = agg_df.merge(label_df, on='co_ref', how='left')

In [13]:
# Fill numeric features with 0; keep target/category columns as strings
num_cols = agg_df.select_dtypes(include=['number']).columns
agg_df[num_cols] = agg_df[num_cols].fillna(0)

if 'payment_method_mode' in agg_df.columns:
    agg_df['payment_method_mode'] = agg_df['payment_method_mode'].fillna('unknown')

if 'prospect_outcome' in agg_df.columns:
    agg_df['prospect_outcome'] = agg_df['prospect_outcome'].fillna('Unknown')

In [14]:
agg_df.to_csv("../../data/03_final/final_billings_features.csv", index=False)